# Benchmark: Run MSA Tools on BAliBASE 3.0

Runs all 4 aligner configurations on every BAliBASE problem and records:
- **SP score** and **TC score** (computed against the reference MSF alignment)
- **Runtime** (wall-clock seconds)
- **Peak memory** (MB, via `/usr/bin/time -v`)

Results are saved to `results/results.csv`. The loop is **resumable**: already-computed rows are skipped.

## 1. Imports & Configuration

In [14]:
import subprocess, re, time, os
from pathlib import Path
from collections import OrderedDict

import pandas as pd
from tqdm.notebook import tqdm
from Bio import AlignIO

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_DIR = Path(r'C:\Users\anand\Desktop\SEM 4\CS 502\Project')
BB3_DIR     = PROJECT_DIR / 'data' / 'bb3_release'
ALN_DIR     = PROJECT_DIR / 'results' / 'alignments'
CSV_PATH    = PROJECT_DIR / 'results' / 'results.csv'

RV_SETS = ['RV11', 'RV12', 'RV20', 'RV30', 'RV40', 'RV50']

ALIGNERS = {
    'mafft_fftns2': 'mafft --retree 2 --thread 1 {input} > {output}',
    'mafft_linsi':  'mafft --maxiterate 1000 --localpair --thread 1 {input} > {output}',
    'muscle5':      'muscle5 -align {input} -output {output}',
    'clustalo':     'clustalo -i {input} -o {output} --force',
    'kalign3':      'kalign -i {input} -o {output}',
    # t_coffee cannot parse paths with spaces; run_aligner copies to /tmp first.
    't_coffee':     't_coffee {input} -outfile {output} -output fasta_aln -quiet',
    'famsa':        'famsa {input} {output}',
}

ALN_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Config OK — aligners:', list(ALIGNERS.keys()))

Config OK — aligners: ['mafft_fftns2', 'mafft_linsi', 'muscle5', 'clustalo', 'kalign3', 't_coffee', 'famsa']


## 2. Helper: Windows ↔ WSL Path Conversion

In [15]:
def to_wsl(win_path):
    """Convert a Windows absolute path to its WSL /mnt/... equivalent."""
    p = str(win_path).replace('\\', '/')
    if len(p) >= 2 and p[1] == ':':
        p = f'/mnt/{p[0].lower()}{p[2:]}'
    return p

## 3. Helper: Run Aligner & Measure Time/Memory

In [16]:
def run_aligner(aligner_name, input_fasta: Path, output_fasta: Path, timeout=120):
    """
    Run an MSA tool via WSL with a hard timeout.
    Returns (runtime_sec, peak_mem_mb, success: bool).

    stdout/stderr are discarded (DEVNULL) so that killing the WSL process
    on timeout does not leave orphan children holding pipe file descriptors
    open — which would cause proc.wait() to block indefinitely.
    """
    wsl_in   = to_wsl(input_fasta)
    wsl_out  = to_wsl(output_fasta)
    wsl_time = wsl_out + '.time'

    # t_coffee cannot handle paths with spaces; copy input to /tmp first.
    if aligner_name == 't_coffee':
        import os
        stem    = input_fasta.stem or input_fasta.name
        tmp_in  = f'/tmp/tc_{stem}_{os.getpid()}.fasta'
        tmp_out = f'/tmp/tc_{stem}_{os.getpid()}_out.fasta'
        subprocess.run(['wsl', '-e', 'bash', '-c',
                        f'cp "{wsl_in}" {tmp_in}'],
                       capture_output=True)
        cmd_in, cmd_out = tmp_in, tmp_out
    else:
        cmd_in, cmd_out = wsl_in, wsl_out

    aln_cmd  = ALIGNERS[aligner_name].format(input=f"'{cmd_in}'", output=f"'{cmd_out}'")
    full_cmd = (f"/usr/bin/time -v -o '{wsl_time}' bash -c "
                f'"{aln_cmd} 2>/dev/null"')

    t0   = time.perf_counter()
    proc = subprocess.Popen(
        ['wsl', '-e', 'bash', '-c', full_cmd],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    try:
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()   # reap the zombie; no pipe to drain
        # Kill lingering WSL child processes by name
        subprocess.run(
            ['wsl', '-e', 'bash', '-c',
             'pkill -9 -f "t_coffee"; pkill -9 -f "kalign"; pkill -9 -f "famsa"'],
            capture_output=True
        )
        raise

    elapsed = time.perf_counter() - t0

    # Copy t_coffee output from /tmp back to real path, then clean up
    if aligner_name == 't_coffee':
        subprocess.run(['wsl', '-e', 'bash', '-c',
                        f'[ -f {tmp_out} ] && cp {tmp_out} "{wsl_out}"; '
                        f'rm -f {tmp_in} {tmp_out}'],
                       capture_output=True)

    # Brief pause so WSL filesystem writes are visible to Windows
    time.sleep(0.2)

    success = (proc.returncode == 0
               and output_fasta.exists()
               and output_fasta.stat().st_size > 0)

    peak_mb   = None
    time_file = Path(str(output_fasta) + '.time')
    if time_file.exists():
        m = re.search(r'Maximum resident set size \(kbytes\): (\d+)', time_file.read_text())
        if m:
            peak_mb = int(m.group(1)) / 1024

    return elapsed, peak_mb, success

## 4. Helper: SP & TC Scoring
Python re-implementation of the `bali_score` utility.

- **SP (Sum-of-Pairs)**: fraction of residue pairs that are correctly co-aligned relative to the reference.
- **TC (Total Column)**: fraction of reference columns that are completely reproduced in the test alignment.

In [17]:
GAP_CHARS = set('-. ~')

def parse_msf(path):
    """Read a BAliBASE MSF file. Returns OrderedDict {name: uppercase_seq}."""
    aln = AlignIO.read(str(path), 'msf')
    return OrderedDict((r.id, str(r.seq).upper()) for r in aln)

def parse_fasta_aln(path):
    """Read an aligned FASTA file. Returns OrderedDict {name: uppercase_seq}."""
    aln = AlignIO.read(str(path), 'fasta')
    return OrderedDict((r.id, str(r.seq).upper()) for r in aln)

def normalize_id(s):
    """Strip position suffixes like '/1-286' added by some aligners."""
    return s.split('/')[0].strip()

def match_seqs(ref, test):
    """
    Re-key the test alignment so its keys match the reference exactly.
    Matches by normalizing IDs and falling back to prefix matching.
    """
    norm_test = {normalize_id(k): v for k, v in test.items()}
    matched = OrderedDict()
    for ref_name in ref:
        norm_ref = normalize_id(ref_name)
        if norm_ref in norm_test:
            matched[ref_name] = norm_test[norm_ref]
        else:
            # Prefix fallback
            hits = [k for k in norm_test if k.startswith(norm_ref) or norm_ref.startswith(k)]
            if hits:
                matched[ref_name] = norm_test[hits[0]]
    return matched

def _build_maps(aln):
    """
    For each sequence in the alignment, build:
      - res_to_col[name]: list where index i gives the column of the i-th residue
      - col_to_res[name]: dict mapping column -> residue index
    """
    res_to_col, col_to_res = {}, {}
    for name, seq in aln.items():
        r2c, c2r = [], {}
        ri = 0
        for ci, aa in enumerate(seq):
            if aa not in GAP_CHARS:
                r2c.append(ci)
                c2r[ci] = ri
                ri += 1
        res_to_col[name] = r2c
        col_to_res[name] = c2r
    return res_to_col, col_to_res

def compute_scores(ref_aln, test_aln):
    """
    Compute SP and TC scores.
    Both alignments must cover the same sequences (same keys, same residues).
    Returns (sp: float, tc: float).
    """
    names    = list(ref_aln.keys())
    ref_len  = len(next(iter(ref_aln.values())))

    ref_r2c, ref_c2r = _build_maps(ref_aln)
    tst_r2c, _       = _build_maps(test_aln)

    correct_pairs = total_pairs = 0
    correct_cols  = total_cols  = 0

    for col in range(ref_len):
        # Sequences that have a residue (non-gap) in this reference column
        present = [(n, ref_c2r[n][col]) for n in names if col in ref_c2r[n]]
        if len(present) < 2:
            continue

        # TC: all residues in 'present' must land on the same column in the test alignment
        total_cols += 1
        test_cols_used = set()
        valid = True
        for n, ri in present:
            if ri < len(tst_r2c[n]):
                test_cols_used.add(tst_r2c[n][ri])
            else:
                valid = False
                break
        if valid and len(test_cols_used) == 1:
            correct_cols += 1

        # SP: for each pair, check if they co-occur in the test alignment
        for i in range(len(present)):
            for j in range(i + 1, len(present)):
                ni, ri = present[i]
                nj, rj = present[j]
                total_pairs += 1
                if (ri < len(tst_r2c[ni]) and rj < len(tst_r2c[nj])
                        and tst_r2c[ni][ri] == tst_r2c[nj][rj]):
                    correct_pairs += 1

    sp = correct_pairs / total_pairs if total_pairs > 0 else 0.0
    tc = correct_cols  / total_cols  if total_cols  > 0 else 0.0
    return round(sp, 6), round(tc, 6)

## 5. Helper: Process One Alignment Problem

In [18]:
def process_problem(rv_set, problem_id, tfa_path, msf_path):
    """
    Run all 4 aligners on one BAliBASE problem.
    Returns a list of result dicts (one per aligner).
    """
    ref_aln = parse_msf(msf_path)   # read reference once
    rows = []

    for aligner_name in ALIGNERS:
        out_dir = ALN_DIR / rv_set / aligner_name
        out_dir.mkdir(parents=True, exist_ok=True)
        out_fasta = out_dir / f'{problem_id}.fasta'

        sp = tc = runtime = peak_mb = None
        success = False

        try:
            runtime, peak_mb, success = run_aligner(aligner_name, tfa_path, out_fasta)
            if success:
                test_aln = parse_fasta_aln(out_fasta)
                test_aln = match_seqs(ref_aln, test_aln)
                if len(test_aln) == len(ref_aln):
                    sp, tc = compute_scores(ref_aln, test_aln)
                else:
                    print(f'    {aligner_name}/{problem_id}: seq count mismatch '
                          f'(ref={len(ref_aln)}, test={len(test_aln)})')
        except subprocess.TimeoutExpired:
            print(f'    TIMEOUT: {aligner_name}/{problem_id}')
        except Exception as e:
            print(f'    ERROR: {aligner_name}/{problem_id}: {e}')

        rows.append({
            'rv_set':      rv_set,
            'problem':     problem_id,
            'aligner':     aligner_name,
            'sp_score':    sp,
            'tc_score':    tc,
            'runtime_sec': runtime,
            'peak_mem_mb': peak_mb,
            'success':     success,
        })

    return rows

## 6. Main Benchmark Loop
Skips any (problem, aligner) pair already present in the CSV (checkpointing).

In [19]:
# Load any previously computed results
if CSV_PATH.exists():
    df_existing = pd.read_csv(CSV_PATH)
    done = set(zip(df_existing['problem'], df_existing['aligner']))
    print(f'Resuming: {len(df_existing)} rows already in CSV.')
else:
    df_existing = pd.DataFrame()
    done = set()

new_rows = []

for rv_set in RV_SETS:
    rv_dir = BB3_DIR / rv_set
    if not rv_dir.exists():
        print(f'Skipping {rv_set} (directory not found)')
        continue

    tfa_files = sorted(rv_dir.glob('*.tfa'))
    print(f'\n{rv_set}: {len(tfa_files)} problems')

    for tfa_path in tqdm(tfa_files, desc=rv_set, leave=False):
        prob_id  = tfa_path.stem
        msf_path = rv_dir / f'{prob_id}.msf'

        if not msf_path.exists():
            print(f'  No reference MSF for {prob_id}, skipping.')
            continue

        # Skip if all 4 aligners already done
        remaining = [a for a in ALIGNERS if (prob_id, a) not in done]
        if not remaining:
            continue

        print(f'  {prob_id} ({len(remaining)} aligners remaining)...')
        rows = process_problem(rv_set, prob_id, tfa_path, msf_path)

        # Only keep newly computed rows
        for r in rows:
            if (r['problem'], r['aligner']) not in done:
                new_rows.append(r)

        # Save incrementally every problem
        if new_rows:
            df_all = pd.concat([df_existing, pd.DataFrame(new_rows)], ignore_index=True)
            df_all.to_csv(CSV_PATH, index=False)

print(f'\nBenchmark complete. Results: {CSV_PATH}')

Resuming: 2702 rows already in CSV.

RV11: 76 problems


RV11:   0%|          | 0/76 [00:00<?, ?it/s]


RV12: 88 problems


RV12:   0%|          | 0/88 [00:00<?, ?it/s]


RV20: 82 problems


...


RV40: 49 problems


RV40:   0%|          | 0/49 [00:00<?, ?it/s]


RV50: 31 problems


RV50:   0%|          | 0/31 [00:00<?, ?it/s]


Benchmark complete. Results: C:\Users\anand\Desktop\SEM 4\CS 502\Project\results\results.csv


## 7. Quick Summary

In [20]:
df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df)}')
print(f'Success rate by aligner:')
print(df.groupby('aligner')['success'].mean().round(3).to_string())
print()
print('Mean SP / TC by aligner (successful runs only):')
ok = df[df['success'] == True]
print(ok.groupby('aligner')[['sp_score', 'tc_score', 'runtime_sec', 'peak_mem_mb']]
        .mean().round(4).to_string())

Total rows: 2702
Success rate by aligner:
aligner
clustalo        1.000
famsa           1.000
kalign3         1.000
mafft_fftns2    1.000
mafft_linsi     1.000
muscle5         1.000
t_coffee        0.886

Mean SP / TC by aligner (successful runs only):
              sp_score  tc_score  runtime_sec  peak_mem_mb
aligner                                                   
clustalo        0.7178    0.3625       1.6741      38.9716
famsa           0.7184    0.3570       0.4043      22.2632
kalign3         0.6921    0.3223       0.0736       5.4668
mafft_fftns2    0.6883    0.3151       0.6125      28.2443
mafft_linsi     0.7425    0.3785       4.3785      27.5653
muscle5         0.7614    0.3986       3.9156     213.6685
t_coffee        0.7455    0.3943      17.4008     553.5504


---
## Cross-Dataset Benchmark: PREFAB v4 · OXBench · SABRE

Extends the BAliBASE 3.0 results to three additional datasets from the drive5
benchmark suite (`data/bench_datasets/bench1.0/`).

All three use a uniform FASTA format where **uppercase letters mark scored
columns** — only residue pairs that are uppercase in the reference count toward
the SP and TC scores (equivalent to the Q-score metric used in MUSCLE5 and
Clustal Omega papers).

| Dataset | Problems | Seqs/problem | Notes |
|---------|----------|--------------|-------|
| PREFAB v4 | 1 681 | ~44 | Pairwise-based; used in MUSCLE5 paper |
| OXBench (ox) | 395 | ~5 | Structural ground truth |
| SABRE | 423 | ~4 | Consistent structural superpositions |

Results are written to separate CSVs so BAliBASE scores remain untouched.


In [21]:
# ── Scoring helpers for drive5 uppercase-mask format ─────────────────────────

def parse_fasta_ref_raw(path):
    """
    Read a drive5-format FASTA reference alignment, preserving case.
    Uppercase residues mark scored columns; lowercase and '.' are unscored.
    Returns OrderedDict {seq_name: aligned_sequence_with_original_case}.
    """
    aln = AlignIO.read(str(path), 'fasta')
    return OrderedDict(
        (r.id.split('/')[0].strip(), str(r.seq))
        for r in aln
    )


def compute_scores_masked(ref_aln_raw, test_aln):
    """
    SP and TC scores for drive5-format benchmarks.

    Only positions that are uppercase in the reference count toward the score:
      - SP: fraction of uppercase residue pairs correctly co-aligned in the test.
      - TC: fraction of columns where every sequence has an uppercase residue
            AND all those residues land in the same test column.

    Parameters
    ----------
    ref_aln_raw : OrderedDict {name: mixed-case aligned sequence}
    test_aln    : OrderedDict {name: uppercase aligned sequence}
    """
    GAP   = set('-. ~')
    names = list(ref_aln_raw.keys())

    # Build per-sequence maps from the reference.
    ref_c2r   = {}   # col -> residue_index (all residues, incl. lowercase)
    ref_upper = {}   # name -> set of residue indices that are uppercase

    for name, seq in ref_aln_raw.items():
        c2r, upper = {}, set()
        ri = 0
        for ci, aa in enumerate(seq):
            if aa not in GAP:
                c2r[ci] = ri
                if aa.isupper():
                    upper.add(ri)
                ri += 1
        ref_c2r[name]   = c2r
        ref_upper[name] = upper

    # Build residue-to-column maps from the test alignment.
    tst_r2c = {}
    for name, seq in test_aln.items():
        r2c = []
        for ci, aa in enumerate(seq):
            if aa not in GAP:
                r2c.append(ci)
        tst_r2c[name] = r2c

    ref_len = len(next(iter(ref_aln_raw.values())))

    # SP score ----------------------------------------------------------------
    cp = tp = 0
    for col in range(ref_len):
        # Pairs where both residues are uppercase in the reference
        scored = [
            (n, ref_c2r[n][col])
            for n in names
            if col in ref_c2r[n] and ref_c2r[n][col] in ref_upper[n]
        ]
        if len(scored) < 2:
            continue
        for i in range(len(scored)):
            for j in range(i + 1, len(scored)):
                ni, ri = scored[i]
                nj, rj = scored[j]
                tp += 1
                if (ri < len(tst_r2c.get(ni, [])) and
                        rj < len(tst_r2c.get(nj, [])) and
                        tst_r2c[ni][ri] == tst_r2c[nj][rj]):
                    cp += 1

    # TC score ----------------------------------------------------------------
    cc = tc_ = 0
    for col in range(ref_len):
        scored = [
            (n, ref_c2r[n][col])
            for n in names
            if col in ref_c2r[n] and ref_c2r[n][col] in ref_upper[n]
        ]
        # TC requires ALL sequences to have an uppercase residue at this column
        if len(scored) != len(names):
            continue
        tc_ += 1
        test_cols, valid = set(), True
        for n, ri in scored:
            if ri < len(tst_r2c.get(n, [])):
                test_cols.add(tst_r2c[n][ri])
            else:
                valid = False
                break
        if valid and len(test_cols) == 1:
            cc += 1

    return round(cp / tp if tp else 0, 6), round(cc / tc_ if tc_ else 0, 6)


print('Uppercase-mask scoring helpers loaded.')


Uppercase-mask scoring helpers loaded.


In [22]:
def run_bench_dataset(dataset_name, in_dir, ref_dir, csv_path,
                      timeout=180, max_problems=None):
    """
    Benchmark all 7 aligners on a drive5-format dataset.

    Parameters
    ----------
    dataset_name  : label used in the CSV 'dataset' column
    in_dir        : Path to input FASTA files (no extension)
    ref_dir       : Path to reference FASTA files (uppercase = scored columns)
    csv_path      : Path to the output CSV (appended if it exists)
    timeout       : seconds per aligner call before it is killed
    max_problems  : if set, only process the first N problems (useful for testing)
    """
    in_dir, ref_dir, csv_path = Path(in_dir), Path(ref_dir), Path(csv_path)
    aln_base = PROJECT_DIR / 'results' / 'alignments_bench' / dataset_name
    aln_base.mkdir(parents=True, exist_ok=True)

    fieldnames = ['dataset', 'problem', 'aligner',
                  'sp_score', 'tc_score', 'runtime_sec', 'peak_mem_mb', 'success']

    # Load checkpoint
    if csv_path.exists():
        df_ex = pd.read_csv(csv_path)
        done  = set(zip(df_ex['problem'], df_ex['aligner']))
        print(f'Resuming {dataset_name}: {len(df_ex)} rows already done.')
    else:
        done = set()
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        with open(csv_path, 'w', newline='', encoding='utf-8') as fh:
            import csv as _csv
            _csv.DictWriter(fh, fieldnames=fieldnames).writeheader()

    problems = sorted(in_dir.iterdir())
    if max_problems:
        problems = problems[:max_problems]

    print(f'{dataset_name}: {len(problems)} problems, timeout={timeout}s')

    import csv as _csv
    out_fh = open(csv_path, 'a', newline='', encoding='utf-8')
    writer = _csv.DictWriter(out_fh, fieldnames=fieldnames)

    for in_path in tqdm(problems, desc=dataset_name):
        prob_id  = in_path.name            # no extension in drive5 format
        ref_path = ref_dir / prob_id
        if not ref_path.exists():
            continue

        remaining = [a for a in ALIGNERS if (prob_id, a) not in done]
        if not remaining:
            continue

        try:
            ref_aln_raw = parse_fasta_ref_raw(ref_path)
        except Exception as e:
            print(f'  Cannot parse reference for {prob_id}: {e}')
            continue

        for aligner_name in remaining:
            aln_dir   = aln_base / aligner_name
            aln_dir.mkdir(parents=True, exist_ok=True)
            out_fasta = aln_dir / f'{prob_id}.fasta'

            sp = tc = runtime = peak_mb = None
            success = False
            try:
                runtime, peak_mb, success = run_aligner(
                    aligner_name, in_path, out_fasta, timeout=timeout)
                if success:
                    test_aln = parse_fasta_aln(out_fasta)
                    test_aln = match_seqs(
                        OrderedDict((k, v) for k, v in ref_aln_raw.items()),
                        test_aln)
                    if len(test_aln) == len(ref_aln_raw):
                        # Use uppercase-aware scoring
                        matched_raw = OrderedDict(
                            (k, ref_aln_raw[k]) for k in test_aln)
                        sp, tc = compute_scores_masked(matched_raw, test_aln)
                    else:
                        print(f'  seq mismatch {aligner_name}/{prob_id} '
                              f'(ref={len(ref_aln_raw)}, test={len(test_aln)})')
            except subprocess.TimeoutExpired:
                print(f'  TIMEOUT: {aligner_name}/{prob_id}')
            except Exception as e:
                print(f'  ERROR: {aligner_name}/{prob_id}: {e}')

            writer.writerow({
                'dataset': dataset_name, 'problem': prob_id,
                'aligner': aligner_name, 'sp_score': sp, 'tc_score': tc,
                'runtime_sec': runtime, 'peak_mem_mb': peak_mb, 'success': success,
            })
            out_fh.flush()
            done.add((prob_id, aligner_name))

    out_fh.close()
    df = pd.read_csv(csv_path)
    ok = df['success'].eq(True).sum()
    print(f'Done. {len(df)} rows written, {ok} successful.')
    return df


print('Generic dataset runner loaded.')


Generic dataset runner loaded.


### Run PREFAB v4

1 681 problems — this is the largest dataset. T-Coffee and MAFFT L-INS-i will
be slow on problems with many sequences. Run with `max_problems=None` for the
full benchmark, or pass a number (e.g. `max_problems=200`) for a quick test.


In [23]:
BENCH_BASE = PROJECT_DIR / 'data' / 'bench_datasets' / 'bench1.0'

df_prefab = run_bench_dataset(
    dataset_name = 'prefab4',
    in_dir       = BENCH_BASE / 'prefab4' / 'in',
    ref_dir      = BENCH_BASE / 'prefab4' / 'ref',
    csv_path     = PROJECT_DIR / 'results' / 'results_prefab4.csv',
    timeout      = 180,
    max_problems = None,   # set to e.g. 200 for a quick test
)


Resuming prefab4: 11767 rows already done.
prefab4: 1681 problems, timeout=180s


prefab4:   0%|          | 0/1681 [00:00<?, ?it/s]

Done. 11767 rows written, 10370 successful.


### Run OXBench

395 problems, ~5 sequences each — fast.

In [24]:
df_ox = run_bench_dataset(
    dataset_name = 'oxbench',
    in_dir       = BENCH_BASE / 'ox' / 'in',
    ref_dir      = BENCH_BASE / 'ox' / 'ref',
    csv_path     = PROJECT_DIR / 'results' / 'results_oxbench.csv',
    timeout      = 120,
    max_problems = None,
)


Resuming oxbench: 2765 rows already done.
oxbench: 395 problems, timeout=120s


oxbench:   0%|          | 0/395 [00:00<?, ?it/s]

Done. 2765 rows written, 2764 successful.


### Run SABRE

423 problems, ~4 sequences each — fast.

In [25]:
df_sabre = run_bench_dataset(
    dataset_name = 'sabre',
    in_dir       = BENCH_BASE / 'sabre' / 'in',
    ref_dir      = BENCH_BASE / 'sabre' / 'ref',
    csv_path     = PROJECT_DIR / 'results' / 'results_sabre.csv',
    timeout      = 120,
    max_problems = None,
)


Resuming sabre: 2961 rows already done.
sabre: 423 problems, timeout=120s


sabre:   0%|          | 0/423 [00:00<?, ?it/s]

Done. 2961 rows written, 2959 successful.


### Cross-Dataset Summary

In [26]:
import os

summary_rows = []
for csv_name, label in [('results_prefab4.csv', 'PREFAB v4'),
                         ('results_oxbench.csv', 'OXBench'),
                         ('results_sabre.csv',   'SABRE')]:
    p = PROJECT_DIR / 'results' / csv_name
    if not p.exists():
        continue
    df = pd.read_csv(p)
    df_ok = df[df['success'] == True]
    for aligner in ALIGNERS:
        sub = df_ok[df_ok['aligner'] == aligner]
        if sub.empty:
            continue
        summary_rows.append({
            'dataset':        label,
            'aligner':        aligner,
            'n_problems':     len(sub),
            'sp_mean':        round(sub['sp_score'].mean(), 4),
            'tc_mean':        round(sub['tc_score'].mean(), 4),
            'runtime_median': round(sub['runtime_sec'].median(), 3),
        })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print(df_summary.to_string(index=False))
else:
    print('No cross-dataset results yet — run the benchmark cells above first.')


  dataset      aligner  n_problems  sp_mean  tc_mean  runtime_median
PREFAB v4 mafft_fftns2        1681   0.6504   0.6504           0.579
PREFAB v4  mafft_linsi        1681   0.6942   0.6942           1.611
PREFAB v4      muscle5        1681   0.6904   0.6904           7.847
PREFAB v4     clustalo         369   0.6957   0.6957          27.293
PREFAB v4      kalign3        1681   0.6169   0.6169           0.193
PREFAB v4     t_coffee        1596   0.6799   0.6799          32.219
PREFAB v4        famsa        1681   0.6586   0.6586           0.252
  OXBench mafft_fftns2         395   0.8819   0.7841           0.483
  OXBench  mafft_linsi         395   0.8858   0.7928           0.463
  OXBench      muscle5         395   0.8978   0.8110           0.189
  OXBench     clustalo         394   0.8889   0.8000           0.265
  OXBench      kalign3         395   0.8856   0.7896           0.173
  OXBench     t_coffee         395   0.8978   0.8113           1.227
  OXBench        famsa         395